# CAT-LM test generation (Colab, free T4 GPU)

Part of the CANDOR-D4J baselines (see `scripts/run_catlm.py`).

1. **Runtime → Change runtime type → T4 GPU.**
2. Run all cells. The upload cell asks for `prompts.jsonl` (from `results/catlm/prompts.jsonl`).
3. At the end `generations.jsonl` downloads. Put it at `results/catlm/generations.jsonl` and run
   `python3 scripts/run_catlm.py evaluate bugs/dev40.txt` on the Mac.

Settings = the CAT-LM paper / authors' `main_sample.py`: 10 samples per prompt, temperature 0.2,
top_k 50, top_p 0.95, max 512 new tokens, trailing `</s>` removed. Generation stops early once the
test method's braces close (only saves time; the Mac cuts each sample at the same point).
If the Colab session disconnects, just run all cells again: finished prompts are skipped.
No API keys or personal data are needed.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# use Colab's own transformers (an old pinned version has no ready wheel for Colab's Python)
!pip -q install sentencepiece accelerate
import torch, transformers
print('transformers', transformers.__version__, '| torch', torch.__version__)
assert torch.cuda.is_available(), ("NO GPU in this session. Runtime -> Change runtime type -> T4 GPU -> Save, "
                                   "then Runtime -> Run all again.")
print('GPU OK:', torch.cuda.get_device_name(0))

In [ ]:
import os, json
from google.colab import files
if not os.path.exists('prompts.jsonl'):
    files.upload()          # choose prompts.jsonl
prompts = [json.loads(l) for l in open('prompts.jsonl')]
todo = [p for p in prompts if 'prompt_full' in p]
print(len(prompts), 'classes,', len(todo), 'with a prompt')

In [ ]:
import torch, time, transformers
import sentencepiece as spm
from huggingface_hub import hf_hub_download
from transformers import AutoModelForCausalLM, StoppingCriteria, StoppingCriteriaList
# Tokenizer: CAT-LM's own sentencepiece model, used directly. The model card's T5Tokenizer wrapper
# does not load in transformers 5; plain sentencepiece gives the SAME token ids (checked on all
# 98 prompts against the original slow tokenizer, minus its trailing </s>, which CAT-LM never saw).
sp = spm.SentencePieceProcessor(model_file=hf_hub_download('nikitharao/catlm', 'spiece.model'))
EOS, PAD = 1, 2                                    # <|endoftext|>, <|padding|> (model config)
dt = {'dtype' if int(transformers.__version__.split('.')[0]) >= 5 else 'torch_dtype': torch.float16}
model = AutoModelForCausalLM.from_pretrained('nikitharao/catlm', use_safetensors=False, **dt).cuda().eval()
N_SAMPLES, MAX_NEW, WINDOW = 10, 512, 8192
KV_BYTES_PER_TOKEN = 2 * model.config.num_hidden_layers * model.config.hidden_size * 2   # K and V, fp16
print('loaded', model.config.model_type, model.dtype)

In [ ]:
def method_closed(text):
    """True once 'public void test' + text has a complete method (same rule as run_catlm.first_method)."""
    text = "public void test" + text
    depth, opened, i, n = 0, False, 0, len(text)
    while i < n:
        c = text[i]
        if text.startswith("//", i):
            i = text.find("\n", i); i = n if i < 0 else i; continue
        if text.startswith("/*", i):
            j = text.find("*/", i + 2); i = n if j < 0 else j + 2; continue
        if c in "\"'":
            j = i + 1
            while j < n and text[j] != c:
                j += 2 if text[j] == "\\" else 1
            i = j + 1; continue
        if c == "{": depth, opened = depth + 1, True
        elif c == "}":
            depth -= 1
            if opened and depth == 0: return True
        i += 1
    return False

def gen_text(prompt_ids, gen_ids):
    """Text of the generated part only. Stops at <|endoftext|>, drops padding. Decoded together with
    the end of the prompt so the spacing at the join is exact."""
    gen = []
    for t in gen_ids:
        if t == EOS: break
        if t != PAD: gen.append(t)
    tail = prompt_ids[-32:]
    return sp.decode(tail + gen)[len(sp.decode(tail)):]

class MethodClosed(StoppingCriteria):
    def __init__(self, prompt_ids): self.p = prompt_ids
    def __call__(self, input_ids, scores, **kw):
        L = len(self.p)
        return torch.tensor([method_closed(gen_text(self.p, row[L:].tolist())) for row in input_ids],
                            dtype=torch.bool, device=input_ids.device)

def encode(text):
    return sp.encode(text)                         # list of ids, no </s>

In [ ]:
done = set()
if os.path.exists('generations.jsonl'):
    done = {(g['bug'], g['class']) for g in map(json.loads, open('generations.jsonl'))}
limit = WINDOW - MAX_NEW
t_all = time.time()
for k, p in enumerate(todo):
    if (p['bug'], p['class']) in done:
        continue
    t = time.time()
    pids, used = encode(p['prompt_full']), 'full'
    if len(pids) > limit:
        pids, used = encode(p['prompt_short']), 'short'
    if len(pids) > limit:
        pids, used = pids[-limit:], 'short_truncated'
    ids = torch.tensor([pids], device='cuda')
    L = len(pids)
    free = torch.cuda.mem_get_info()[0] * 0.6
    per_call = max(1, min(N_SAMPLES, int(free // (KV_BYTES_PER_TOKEN * (L + MAX_NEW)))))
    samples = []
    torch.manual_seed(1000 + k)                    # fixed seed per prompt: repeatable
    while len(samples) < N_SAMPLES:
        n = min(per_call, N_SAMPLES - len(samples))
        with torch.no_grad():
            out = model.generate(ids, attention_mask=torch.ones_like(ids), do_sample=True,
                                 max_new_tokens=MAX_NEW, top_k=50, top_p=0.95, temperature=0.2,
                                 num_return_sequences=n, eos_token_id=EOS, pad_token_id=PAD,
                                 stopping_criteria=StoppingCriteriaList([MethodClosed(pids)]))
        samples += [gen_text(pids, o[L:].tolist()) for o in out]
        torch.cuda.empty_cache()
    g = {'bug': p['bug'], 'class': p['class'], 'prompt_used': used, 'prompt_tokens': L,
         'samples': samples, 'gen_seconds': round(time.time() - t, 1)}
    with open('generations.jsonl', 'a') as f:
        f.write(json.dumps(g) + '\n')
    print(f"{k+1}/{len(todo)} {p['bug']} {p['class'].split('.')[-1]}: {used}, {L} tokens, "
          f"{g['gen_seconds']}s, closed {sum(method_closed(s) for s in samples)}/{N_SAMPLES}", flush=True)
print('all done in', round((time.time() - t_all) / 60, 1), 'min')

In [ ]:
files.download('generations.jsonl')